# 09. CatBoostClassifier и Top-12

Notebook загружает три готовые ranking tables. Здесь нет ALS, Content-Based, candidate generation или повторного feature engineering.

## Подключение проекта

**Что делаем:** определяем корень проекта.  
**Зачем:** одинаковые пути должны работать локально и в Colab.  
**Что получим:** `PROJECT_ROOT` и доступный пакет из `src`.

In [1]:
from pathlib import Path
import sys

try:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_ROOT = Path("/content/drive/MyDrive/fashion-recommender-system")
except ImportError:
    PROJECT_ROOT = Path.cwd().resolve()

if not (PROJECT_ROOT / "src").is_dir():
    raise FileNotFoundError(f"Не найдена папка src: {PROJECT_ROOT / 'src'}")
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))
print("Корень проекта:", PROJECT_ROOT)

Корень проекта: <PROJECT_ROOT>


### Зависимости

**Что делаем:** устанавливаем requirements только в Colab.  
**Зачем:** локальное окружение не должно изменяться при каждом запуске.  
**Что получим:** готовые библиотеки для следующих ячеек.

In [2]:
import subprocess

if "google.colab" in sys.modules:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r",
         str(PROJECT_ROOT / "requirements.txt")],
        check=True,
    )

### Импорты

**Что делаем:** подключаем CatBoost, метрики и persistence  
**Зачем:** ranking model является единственной обучаемой моделью notebook  
**Что получим:** необходимые библиотеки

In [3]:
import time
import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
from IPython.display import display

from fashion_recommender.baselines import popular_items as rank_popular_items
from fashion_recommender.data import load_transactions
from fashion_recommender.evaluation import (
    candidate_recall_at_k, hit_rate_at_k, map_at_k, mean_recall_at_k,
)
from fashion_recommender.persistence import (
    load_json, save_catboost_model, save_json, save_recommendations,
)

### Пути и параметры

**Что делаем:** задаём три ranking files и CatBoost settings  
**Зачем:** все входы должны существовать до запуска fit  
**Что получим:** пути и reproducible параметры

In [4]:
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
MODEL_DIR = PROJECT_ROOT / "models"
REPORT_DIR = PROJECT_ROOT / "reports" / "tables"
ARTIFACT_DIR = PROJECT_ROOT / "artifacts"
TRANSACTIONS_PATH = RAW_DIR / "transactions_train.csv"
ARTICLES_PATH = RAW_DIR / "articles.csv"
CUSTOMERS_PATH = RAW_DIR / "customers.csv"
for directory in [PROCESSED_DIR, MODEL_DIR, REPORT_DIR, ARTIFACT_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

WINDOWS_PATH = PROCESSED_DIR / "temporal_windows.json"
TRAIN_PATH = PROCESSED_DIR / "train_ranking_table.parquet"
VALIDATION_PATH = PROCESSED_DIR / "validation_ranking_table.parquet"
TEST_PATH = PROCESSED_DIR / "test_ranking_table.parquet"

RANDOM_STATE = 42
NEGATIVE_RATIO = 10
ITERATIONS = 500
DEPTH = 6
LEARNING_RATE = 0.05
EARLY_STOPPING_ROUNDS = 50

### Проверка ranking artifacts

**Что делаем:** проверяем JSON окон и три Parquet  
**Зачем:** отсутствие файла не должно запускать предыдущие этапы  
**Что получим:** понятную ошибку с notebook 08

In [5]:
required_paths = [
    WINDOWS_PATH,
    TRAIN_PATH,
    VALIDATION_PATH,
    TEST_PATH,
]
missing_paths = [path for path in required_paths if not path.is_file()]
if missing_paths:
    raise FileNotFoundError(
        f"Не найдены ranking artifacts: {missing_paths}. "
        "Сначала выполните notebook 08_feature_engineering_colab.ipynb."
    )

### Загрузка train

**Что делаем:** читаем готовую train ranking table  
**Зачем:** кандидаты, признаки и target уже рассчитаны  
**Что получим:** `train_table`

In [6]:
train_table = pd.read_parquet(TRAIN_PATH)
print("Train:", train_table.shape)
display(train_table.head())

Train: (445008, 42)
                                         customer_id  ... target
0  0056238bf79efd3e95fc0ed4f4c01c018ff088544fa010...  ...      0
1  0056238bf79efd3e95fc0ed4f4c01c018ff088544fa010...  ...      0
2  0056238bf79efd3e95fc0ed4f4c01c018ff088544fa010...  ...      0
3  0056238bf79efd3e95fc0ed4f4c01c018ff088544fa010...  ...      0
4  0056238bf79efd3e95fc0ed4f4c01c018ff088544fa010...  ...      0

[5 rows x 42 columns]


### Загрузка validation

**Что делаем:** читаем validation ranking table  
**Зачем:** она используется только для early stopping  
**Что получим:** `validation_table`

In [7]:
validation_table = pd.read_parquet(VALIDATION_PATH)
print("Validation:", validation_table.shape)
display(validation_table.head())

Validation: (444853, 42)
                                         customer_id  ... target
0  0044c0b989ea44be655875ac6092dd7c86ab21b1218faf...  ...      0
1  0044c0b989ea44be655875ac6092dd7c86ab21b1218faf...  ...      0
2  0044c0b989ea44be655875ac6092dd7c86ab21b1218faf...  ...      0
3  0044c0b989ea44be655875ac6092dd7c86ab21b1218faf...  ...      0
4  0044c0b989ea44be655875ac6092dd7c86ab21b1218faf...  ...      0

[5 rows x 42 columns]


### Загрузка test

**Что делаем:** читаем final test ranking table  
**Зачем:** test не участвует в fit  
**Что получим:** `test_table`

In [8]:
test_table = pd.read_parquet(TEST_PATH)
print("Test:", test_table.shape)
display(test_table.head())

Test: (445123, 42)
                                         customer_id  ... target
0  0056238bf79efd3e95fc0ed4f4c01c018ff088544fa010...  ...      0
1  0056238bf79efd3e95fc0ed4f4c01c018ff088544fa010...  ...      0
2  0056238bf79efd3e95fc0ed4f4c01c018ff088544fa010...  ...      0
3  0056238bf79efd3e95fc0ed4f4c01c018ff088544fa010...  ...      0
4  0056238bf79efd3e95fc0ed4f4c01c018ff088544fa010...  ...      0

[5 rows x 42 columns]


### Проверка target

**Что делаем:** смотрим число positives и долю класса в каждом split  
**Зачем:** пустой класс сделал бы обучение некорректным  
**Что получим:** таблицу class balance

In [9]:
target_summary = pd.DataFrame({
    "split": ["train", "validation", "test"],
    "rows": [len(train_table), len(validation_table), len(test_table)],
    "positives": [
        train_table["target"].sum(),
        validation_table["target"].sum(),
        test_table["target"].sum(),
    ],
    "positive_share": [
        train_table["target"].mean(),
        validation_table["target"].mean(),
        test_table["target"].mean(),
    ],
})
display(target_summary)
assert train_table["target"].nunique() == 2

        split    rows  positives  positive_share
0       train  445008        154        0.000346
1  validation  444853        128        0.000288
2        test  445123        126        0.000283


### Числовые feature columns

**Что делаем:** явно перечисляем user, item, pair и generator signals  
**Зачем:** ID и target не должны попасть в X  
**Что получим:** 33 numeric feature names

In [10]:
NUMERIC_FEATURES = [
    "user_total_purchases", "user_unique_items", "user_active_days",
    "user_days_since_last_purchase", "user_average_price", "user_online_share",
    "user_age", "item_total_purchases", "item_unique_customers",
    "item_popularity_7d", "item_popularity_30d", "item_average_price",
    "item_days_since_last_purchase", "user_bought_item_before",
    "user_item_purchase_count", "days_since_user_bought_item",
    "user_product_type_count", "user_colour_count", "user_section_count",
    "user_garment_group_count", "als_score", "als_rank",
    "content_similarity_score", "content_rank", "personal_history_score",
    "personal_history_rank", "popularity_score", "popularity_rank",
    "from_als", "from_content_based", "from_personal_history",
    "from_popularity", "number_of_candidate_sources",
]

### Категориальные features

**Что делаем:** отдельно перечисляем шесть item categories  
**Зачем:** CatBoost получает их строками без ручного one-hot  
**Что получим:** `CATEGORICAL_FEATURES` и полный список

In [11]:
CATEGORICAL_FEATURES = [
    "product_type_name",
    "product_group_name",
    "colour_group_name",
    "department_name",
    "section_name",
    "garment_group_name",
]
FEATURE_COLUMNS = NUMERIC_FEATURES + CATEGORICAL_FEATURES
print("Всего features:", len(FEATURE_COLUMNS))

Всего features: 39


### Проверка feature contract

**Что делаем:** сравниваем ожидаемые столбцы со всеми tables  
**Зачем:** ошибка schema должна появиться до sampling  
**Что получим:** подтверждённый contract

In [12]:
for table_name, table in [
    ("train", train_table),
    ("validation", validation_table),
    ("test", test_table),
]:
    missing_features = set(FEATURE_COLUMNS) - set(table.columns)
    if missing_features:
        raise ValueError(
            f"В {table_name} отсутствуют features: {sorted(missing_features)}"
        )
print("Feature contract: OK")

Feature contract: OK


### Положительные train pairs

**Что делаем:** выбираем все строки target=1  
**Зачем:** ни один positive не должен потеряться при sampling  
**Что получим:** `positive_train`

In [13]:
positive_train = train_table[
    train_table["target"] == 1
].copy()
print("Positive train rows:", len(positive_train))

Positive train rows: 154


### Отрицательные train pairs

**Что делаем:** отдельно выбираем target=0  
**Зачем:** negative sampling применяется только к train  
**Что получим:** `negative_train`

In [14]:
negative_train = train_table[
    train_table["target"] == 0
].copy()
print("Negative train rows:", len(negative_train))

Negative train rows: 444854


### Negative sampling

**Что делаем:** берём не больше десяти negatives на positive  
**Зачем:** фиксированный seed делает выбор воспроизводимым  
**Что получим:** `sampled_negatives`

In [15]:
number_of_negatives = min(
    len(negative_train),
    len(positive_train) * NEGATIVE_RATIO,
)
sampled_negatives = negative_train.sample(
    n=number_of_negatives,
    random_state=RANDOM_STATE,
)
print("Sampled negatives:", len(sampled_negatives))

Sampled negatives: 1540


### Итоговый train sample

**Что делаем:** объединяем positives и sampled negatives, затем перемешиваем  
**Зачем:** validation и test остаются полными  
**Что получим:** `sampled_train`

In [16]:
sampled_train = pd.concat(
    [positive_train, sampled_negatives],
    ignore_index=True,
)
sampled_train = sampled_train.sample(
    frac=1,
    random_state=RANDOM_STATE,
).reset_index(drop=True)
print("Sampled train:", sampled_train.shape)
print("Positive share:", sampled_train["target"].mean())

Sampled train: (1694, 42)
Positive share: 0.09090909090909091


### X_train

**Что делаем:** выбираем только feature columns  
**Зачем:** customer/article ID не являются model features  
**Что получим:** `X_train`

In [17]:
X_train = sampled_train[FEATURE_COLUMNS].copy()
print("X_train:", X_train.shape)
display(X_train.head())

X_train: (1694, 39)
   user_total_purchases  ...  garment_group_name
0                     6  ...            Swimwear
1                     1  ...            Trousers
2                     1  ...            Swimwear
3                     4  ...            Swimwear
4                     3  ...            Swimwear

[5 rows x 39 columns]


### y_train

**Что делаем:** выбираем target отдельно  
**Зачем:** явное разделение X/y упрощает проверку  
**Что получим:** `y_train`

In [18]:
y_train = sampled_train["target"].copy()
print("y_train:", y_train.shape)
print(y_train.value_counts())

y_train: (1694,)
target
0    1540
1     154
Name: count, dtype: int64


### Validation X/y

**Что делаем:** готовим полный validation split  
**Зачем:** он используется для AUC и early stopping  
**Что получим:** `X_validation` и `y_validation`

In [19]:
X_validation = validation_table[FEATURE_COLUMNS].copy()
y_validation = validation_table["target"].copy()
print("X_validation:", X_validation.shape)
print("y_validation:", y_validation.shape)

X_validation: (444853, 39)
y_validation: (444853,)


### Test X/y

**Что делаем:** готовим final test, не передавая его в fit  
**Зачем:** y_test нужен только для диагностики таблицы  
**Что получим:** `X_test` и `y_test`

In [20]:
X_test = test_table[FEATURE_COLUMNS].copy()
y_test = test_table["target"].copy()
print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

X_test: (445123, 39)
y_test: (445123,)


### Тип категорий

**Что делаем:** приводим шесть category columns к строкам  
**Зачем:** CatBoost должен видеть одинаковый тип во всех splits  
**Что получим:** три согласованных X tables

In [21]:
for column in CATEGORICAL_FEATURES:
    X_train[column] = X_train[column].astype(str)
    X_validation[column] = X_validation[column].astype(str)
    X_test[column] = X_test[column].astype(str)
print(X_train[CATEGORICAL_FEATURES].dtypes)

product_type_name     object
product_group_name    object
colour_group_name     object
department_name       object
section_name          object
garment_group_name    object
dtype: object


### Создание CatBoostClassifier

**Что делаем:** задаём модель и гиперпараметры  
**Зачем:** создание объекта отделено от обучения  
**Что получим:** необученный `model`

In [22]:
model = CatBoostClassifier(
    iterations=ITERATIONS,
    depth=DEPTH,
    learning_rate=LEARNING_RATE,
    loss_function="Logloss",
    eval_metric="AUC",
    random_seed=RANDOM_STATE,
    verbose=50,
    allow_writing_files=False,
)
print(model)

CatBoostClassifier(allow_writing_files=False, depth=6, eval_metric='AUC', iterations=500, learning_rate=0.05, loss_function='Logloss', random_seed=42, verbose=50)


### Обучение CatBoost

**Что делаем:** вызываем `.fit()` с train и validation  
**Зачем:** test не участвует ни в fit, ни в early stopping  
**Что получим:** обученную модель и training time

In [23]:
training_started = time.perf_counter()
model.fit(
    X_train,
    y_train,
    cat_features=CATEGORICAL_FEATURES,
    eval_set=(X_validation, y_validation),
    early_stopping_rounds=EARLY_STOPPING_ROUNDS,
)
training_time = time.perf_counter() - training_started
print("Trees:", model.tree_count_)
print("Training seconds:", round(training_time, 2))

0:	test: 0.7301498	best: 0.7301498 (0)	total: 89.4ms	remaining: 44.6s
50:	test: 0.8182392	best: 0.8222674 (35)	total: 1.14s	remaining: 10s
Stopped by overfitting detector  (50 iterations wait)

bestTest = 0.8222673861
bestIteration = 35

Shrink model to first 36 iterations.
Trees: 36
Training seconds: 2.0


### История обучения

**Что делаем:** читаем AUC по итерациям  
**Зачем:** видим работу early stopping без нового fit  
**Что получим:** последние значения validation AUC

In [24]:
evaluation_history = model.get_evals_result()
validation_auc = evaluation_history["validation"]["AUC"]
history_table = pd.DataFrame({
    "iteration": np.arange(1, len(validation_auc) + 1),
    "validation_auc": validation_auc,
})
display(history_table.tail(10))

    iteration  validation_auc
76         77        0.818339
77         78        0.819820
78         79        0.819472
79         80        0.819517
80         81        0.819522
81         82        0.819628
82         83        0.819957
83         84        0.820261
84         85        0.819447
85         86        0.820497


### Predict probability

**Что делаем:** получаем вероятность positive class для test candidates  
**Зачем:** probability становится ranking score  
**Что получим:** столбец `prediction`

In [25]:
inference_started = time.perf_counter()
test_predictions = model.predict_proba(X_test)[:, 1]
inference_time = time.perf_counter() - inference_started

scored_test = test_table.copy()
scored_test["prediction"] = test_predictions
display(scored_test[["customer_id", "article_id", "prediction", "target"]].head())

                                         customer_id  ... target
0  0056238bf79efd3e95fc0ed4f4c01c018ff088544fa010...  ...      0
1  0056238bf79efd3e95fc0ed4f4c01c018ff088544fa010...  ...      0
2  0056238bf79efd3e95fc0ed4f4c01c018ff088544fa010...  ...      0
3  0056238bf79efd3e95fc0ed4f4c01c018ff088544fa010...  ...      0
4  0056238bf79efd3e95fc0ed4f4c01c018ff088544fa010...  ...      0

[5 rows x 4 columns]


### Сортировка candidates

**Что делаем:** сортируем probability отдельно внутри пользователя  
**Зачем:** высокий score должен иметь меньший rank  
**Что получим:** `ranked_test`

In [26]:
ranked_test = scored_test.sort_values(
    ["customer_id", "prediction", "article_id"],
    ascending=[True, False, True],
).drop_duplicates(["customer_id", "article_id"])
ranked_test["model_rank"] = ranked_test.groupby(
    "customer_id"
).cumcount() + 1
display(ranked_test.head())

                                          customer_id  ... model_rank
19  0056238bf79efd3e95fc0ed4f4c01c018ff088544fa010...  ...          1
21  0056238bf79efd3e95fc0ed4f4c01c018ff088544fa010...  ...          2
15  0056238bf79efd3e95fc0ed4f4c01c018ff088544fa010...  ...          3
13  0056238bf79efd3e95fc0ed4f4c01c018ff088544fa010...  ...          4
24  0056238bf79efd3e95fc0ed4f4c01c018ff088544fa010...  ...          5

[5 rows x 44 columns]


### Первые 12 candidates

**Что делаем:** оставляем model_rank не больше 12  
**Зачем:** fallback добавляется только в следующем шаге  
**Что получим:** `top12_scored`

In [27]:
top12_scored = ranked_test[
    ranked_test["model_rank"] <= 12
].copy()
print("Top candidate rows:", len(top12_scored))
print(top12_scored.groupby("customer_id").size().describe())

Top candidate rows: 24000
count    2000.0
mean       12.0
std         0.0
min        12.0
25%        12.0
50%        12.0
75%        12.0
max        12.0
dtype: float64


### Test history и future

**Что делаем:** загружаем raw transactions только для ground truth и fallback  
**Зачем:** features и candidates здесь не пересчитываются  
**Что получим:** две temporal tables

In [28]:
transactions = load_transactions(TRANSACTIONS_PATH)
windows = load_json(WINDOWS_PATH)
test_cutoff = pd.Timestamp(windows["test"]["cutoff_date"])
test_end = pd.Timestamp(windows["test"]["target_end_date"])
test_history = transactions[
    transactions["t_dat"] < test_cutoff
].copy()
test_future = transactions[
    transactions["t_dat"].between(test_cutoff, test_end)
].copy()
assert test_history["t_dat"].max() < test_cutoff

### Popularity fallback

**Что делаем:** считаем Top-100 только по test history  
**Зачем:** короткие списки дополняются без target information  
**Что получим:** `popular_items`

In [29]:
popular_items = rank_popular_items(
    test_history,
    limit=100,
)["article_id"].tolist()
print("Первые fallback items:", popular_items[:12])

Первые fallback items: ['0706016001', '0706016002', '0372860001', '0464297007', '0706016003', '0759871002', '0562245046', '0673677002', '0673396002', '0568601006', '0562245001', '0156231001']


### Scored lists

**Что делаем:** собираем article/score каждого user из Top-12  
**Зачем:** порядок уже задан model_rank  
**Что получим:** `scored_items_by_user`

In [30]:
scored_items_by_user = {}
for customer_id, group in top12_scored.groupby("customer_id", sort=False):
    scored_items_by_user[customer_id] = list(
        zip(group["article_id"], group["prediction"])
    )
print("Users with scored lists:", len(scored_items_by_user))

Users with scored lists: 2000


### Заполнение Top-12

**Что делаем:** дополняем каждый список популярными товарами без повторов  
**Зачем:** получаем ровно 12 рекомендаций для каждого candidate user  
**Что получим:** `recommendation_rows`

In [31]:
evaluation_users = test_table["customer_id"].drop_duplicates().tolist()
recommendation_rows = []

for customer_id in evaluation_users:
    chosen = list(scored_items_by_user.get(customer_id, []))
    chosen_ids = {article_id for article_id, _ in chosen}
    for article_id in popular_items:
        if article_id not in chosen_ids:
            chosen.append((article_id, 0.0))
            chosen_ids.add(article_id)
        if len(chosen) == 12:
            break
    for rank, (article_id, score) in enumerate(chosen[:12], start=1):
        recommendation_rows.append({
            "customer_id": customer_id,
            "article_id": article_id,
            "rank": rank,
            "score": float(score),
        })

### Итоговая recommendation table

**Что делаем:** создаём DataFrame и проверяем длины списков  
**Зачем:** API ожидает customer, article, rank и score  
**Что получим:** `final_recommendations`

In [32]:
final_recommendations = pd.DataFrame(recommendation_rows)
list_lengths = final_recommendations.groupby("customer_id").size()
print("Recommendations:", final_recommendations.shape)
print("Min/Max list length:", list_lengths.min(), list_lengths.max())
assert list_lengths.eq(12).all()
assert not final_recommendations.duplicated(
    ["customer_id", "article_id"]
).any()
display(final_recommendations.head(12))

Recommendations: (24000, 4)
Min/Max list length: 12 12
                                          customer_id  ...     score
0   0056238bf79efd3e95fc0ed4f4c01c018ff088544fa010...  ...  0.464738
1   0056238bf79efd3e95fc0ed4f4c01c018ff088544fa010...  ...  0.425352
2   0056238bf79efd3e95fc0ed4f4c01c018ff088544fa010...  ...  0.403344
3   0056238bf79efd3e95fc0ed4f4c01c018ff088544fa010...  ...  0.377075
4   0056238bf79efd3e95fc0ed4f4c01c018ff088544fa010...  ...  0.373831
5   0056238bf79efd3e95fc0ed4f4c01c018ff088544fa010...  ...  0.371906
6   0056238bf79efd3e95fc0ed4f4c01c018ff088544fa010...  ...  0.367070
7   0056238bf79efd3e95fc0ed4f4c01c018ff088544fa010...  ...  0.349636
8   0056238bf79efd3e95fc0ed4f4c01c018ff088544fa010...  ...  0.344203
9   0056238bf79efd3e95fc0ed4f4c01c018ff088544fa010...  ...  0.337603
10  0056238bf79efd3e95fc0ed4f4c01c018ff088544fa010...  ...  0.332008
11  0056238bf79efd3e95fc0ed4f4c01c018ff088544fa010...  ...  0.330356

[12 rows x 4 columns]


### Test ground truth

**Что делаем:** собираем полные future items evaluation users  
**Зачем:** denominator Recall не ограничивается candidate positives  
**Что получим:** `test_ground_truth`

In [33]:
test_known_users = set(test_history["customer_id"])
test_future_evaluation = test_future[
    test_future["customer_id"].isin(test_known_users)
    & test_future["customer_id"].isin(evaluation_users)
]
test_future_unique = (
    test_future_evaluation
    .sort_values("t_dat")
    .drop_duplicates(["customer_id", "article_id"])
)
test_ground_truth = test_future_unique.groupby(
    "customer_id", sort=False
)["article_id"].apply(list).to_dict()
print("Ground-truth users:", len(test_ground_truth))

Ground-truth users: 2000


### Dictionaries для метрик

**Что делаем:** преобразуем рекомендации и candidates в списки  
**Зачем:** Top-12 и Candidate Recall используют разные dictionaries  
**Что получим:** два model outputs

In [34]:
recommendations_dict = final_recommendations.sort_values(
    ["customer_id", "rank"]
).groupby("customer_id", sort=False)["article_id"].apply(list).to_dict()

candidate_dict = test_table.groupby(
    "customer_id", sort=False
)["article_id"].apply(list).to_dict()
print("Recommendation users:", len(recommendations_dict))
print("Candidate users:", len(candidate_dict))

Recommendation users: 2000
Candidate users: 2000


### Финальные метрики

**Что делаем:** считаем Recall, MAP, HitRate и Candidate Recall  
**Зачем:** это final test, не участвовавший в fit  
**Что получим:** `final_metrics`

In [35]:
final_metrics = {
    "model": "CatBoost Hybrid",
    "Recall@12": mean_recall_at_k(test_ground_truth, recommendations_dict, 12),
    "MAP@12": map_at_k(test_ground_truth, recommendations_dict, 12),
    "HitRate@12": hit_rate_at_k(test_ground_truth, recommendations_dict, 12),
    "Candidate Recall": candidate_recall_at_k(
        test_ground_truth, candidate_dict, 250
    ),
    "users_evaluated": len(test_ground_truth),
    "average_candidates": test_table.groupby("customer_id").size().mean(),
    "training_time": training_time,
    "inference_time": inference_time,
    "notes": "CatBoostClassifier; common test cohort",
}
display(pd.Series(final_metrics))

model                                        CatBoost Hybrid
Recall@12                                           0.017167
MAP@12                                              0.007842
HitRate@12                                            0.0185
Candidate Recall                                     0.05735
users_evaluated                                         2000
average_candidates                                  222.5615
training_time                                       2.001512
inference_time                                       0.16886
notes                 CatBoostClassifier; common test cohort
dtype: object


### Feature importance

**Что делаем:** получаем важности из уже обученной модели  
**Зачем:** importance описывает использование признака, не причинность  
**Что получим:** отсортированную таблицу

In [36]:
feature_importance = pd.DataFrame({
    "feature": FEATURE_COLUMNS,
    "importance": model.get_feature_importance(),
}).sort_values("importance", ascending=False).reset_index(drop=True)
display(feature_importance.head(20))

                          feature  importance
0             item_popularity_30d   22.547707
1              item_popularity_7d   12.038988
2   item_days_since_last_purchase   11.824070
3              user_average_price    6.269188
4           from_personal_history    5.145515
5           item_unique_customers    5.035972
6              item_average_price    3.465889
7            item_total_purchases    3.267304
8           personal_history_rank    2.596925
9              product_group_name    2.596711
10                      als_score    2.563433
11                       user_age    2.540383
12  user_days_since_last_purchase    2.492855
13              user_online_share    2.420556
14              user_unique_items    1.837072
15               popularity_score    1.558952
16       user_garment_group_count    1.255625
17                       als_rank    1.190138
18              product_type_name    1.183227
19    days_since_user_bought_item    1.139061


### Сохранение CatBoost

**Что делаем:** записываем обученную модель отдельно  
**Зачем:** batch demo загрузит `.cbm` без fit  
**Что получим:** `catboost_recommender.cbm`

In [37]:
catboost_path = save_catboost_model(
    model,
    MODEL_DIR / "catboost_recommender.cbm",
)
print("Сохранено:", catboost_path)

Сохранено: <PROJECT_ROOT>/models/catboost_recommender.cbm


### Сохранение feature config

**Что делаем:** записываем порядок features и categories  
**Зачем:** inference должен передавать колонки в том же порядке  
**Что получим:** два JSON-файла

In [38]:
save_json(FEATURE_COLUMNS, MODEL_DIR / "feature_columns.json")
save_json(CATEGORICAL_FEATURES, MODEL_DIR / "categorical_features.json")
save_json(popular_items, MODEL_DIR / "popular_items.json")
print("Feature config и popularity сохранены")

Feature config и popularity сохранены


### Сохранение metadata

**Что делаем:** записываем фактические даты, cohort и metrics  
**Зачем:** API не должен придумывать сведения о модели  
**Что получим:** `model_metadata.json`

In [39]:
model_metadata = {
    "architecture": "Popularity + Personal History + ALS + Content-Based -> CatBoostClassifier",
    "trained_at": pd.Timestamp.now(tz="UTC"),
    "prediction_horizon_days": 7,
    "recommendation_size": 12,
    "evaluation_user_limit": len(evaluation_users),
    "final_test_metrics": final_metrics,
    "final_test_window_start": test_cutoff,
    "final_test_window_end": test_end,
}
save_json(model_metadata, MODEL_DIR / "model_metadata.json")

### Сохранение рекомендаций

**Что делаем:** записываем готовый serving Parquet  
**Зачем:** API выполняет lookup, а не model inference  
**Что получим:** `final_recommendations.parquet`

In [40]:
recommendations_path = save_recommendations(
    final_recommendations,
    ARTIFACT_DIR / "final_recommendations.parquet",
)
print("Сохранено:", recommendations_path)

Сохранено: <PROJECT_ROOT>/artifacts/final_recommendations.parquet


### Сохранение отчётов

**Что делаем:** записываем importance и test metrics  
**Зачем:** notebook 10 загрузит компактные CSV  
**Что получим:** два report files

In [41]:
feature_importance.to_csv(
    REPORT_DIR / "catboost_feature_importance.csv",
    index=False,
)
pd.DataFrame([final_metrics]).to_csv(
    REPORT_DIR / "catboost_metrics.csv",
    index=False,
)
print("CatBoost reports сохранены")

CatBoost reports сохранены
